# 🧵 FITVISION 3D - TRÍCH XUẤT SỐ ĐO MAY MẶC & TẠO AVATAR 3D TỪ ẢNH (ISO 8559)
Notebook trọn gói chạy trên **Google Colab (GPU T4 miễn phí)**:
- Tự động cài đặt môi trường cô lập Python 3.10 + PyTorch CUDA.
- Tự động tải đủ bộ trọng số AI SMPLer-X & mô hình giải phẫu SMPL-X (Nam, Nữ, Trung tính).
- Trích xuất 13 số đo ISO 8559 và tối ưu hình thể 3D theo Chiều cao - Cân nặng thực tế.

---

### 🟢 BƯỚC 1: Tải mã nguồn dự án (Nhánh QuanHoang)

In [ ]:
# Xóa bản cũ nếu có và clone bản mới nhất từ nhánh QuanHoang
!rm -rf /content/C1SE.12-FitVision3D
!git clone -b QuanHoang https://github.com/le-van-thang/C1SE.12-FitVision3D.git /content/C1SE.12-FitVision3D

%cd /content/C1SE.12-FitVision3D/ai-service
!pwd

### 🟢 BƯỚC 2: Cài đặt môi trường tự động (PyTorch 2.0+cu118, MMCV-Full, SMPL-X, C++ Extension)
*Chạy ô này mất khoảng 2-3 phút. Môi trường sẽ được cấu hình cô lập tại `/opt/py310` để tránh mọi xung đột.*

In [ ]:
%cd /content/C1SE.12-FitVision3D/ai-service
!chmod +x setup_colab.sh
!./setup_colab.sh

### 🟢 BƯỚC 3: Tải trọn bộ 5 mô hình AI & Lưới giải phẫu SMPL-X (Chạy 1 lần duy nhất, ~30 giây)
- `smpler_x_s32.pth.tar` & `vitpose_small.pth`: Mạng Transformer trích xuất tư thế và dáng 3D từ ảnh.
- `SMPLX_NEUTRAL.npz`, `SMPLX_MALE.npz`, `SMPLX_FEMALE.npz`: Lưới giải phẫu 10.475 đỉnh chuẩn.

In [ ]:
%cd /content/C1SE.12-FitVision3D/ai-service

# 1. Tải trọng số mạng SMPLer-X & ViTPose vào pretrained_models
!mkdir -p pretrained_models
!wget -nc -q --show-progress -O pretrained_models/smpler_x_s32.pth.tar \
    https://huggingface.co/caizhongang/SMPLer-X/resolve/main/smpler_x_s32.pth.tar
!wget -nc -q --show-progress -O pretrained_models/vitpose_small.pth \
    https://huggingface.co/camenduru/SMPLer-X/resolve/main/vitpose_small.pth

# 2. Tải đủ bộ file lưới và ma trận khớp SMPL-X vào common/utils/human_model_files/smplx/
!mkdir -p common/utils/human_model_files/smplx
!wget -nc -q --show-progress -O common/utils/human_model_files/smplx/SMPLX_NEUTRAL.npz \
    https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPLX_NEUTRAL.npz
!wget -nc -q --show-progress -O common/utils/human_model_files/smplx/SMPLX_MALE.npz \
    https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPLX_MALE.npz
!wget -nc -q --show-progress -O common/utils/human_model_files/smplx/SMPLX_FEMALE.npz \
    https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPLX_FEMALE.npz
!wget -nc -q --show-progress -O common/utils/human_model_files/smplx/SMPLX_to_J14.pkl \
    https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPLX_to_J14.pkl
!wget -nc -q --show-progress -O common/utils/human_model_files/smplx/MANO_SMPLX_vertex_ids.pkl \
    https://huggingface.co/camenduru/SMPLer-X/resolve/main/MANO_SMPLX_vertex_ids.pkl

print("\n[✓] Kiểm tra danh sách mô hình đã tải thành công:")
!ls -lh pretrained_models/
!ls -lh common/utils/human_model_files/smplx/

### 🟢 BƯỚC 4: Tải ảnh khách hàng cần đo lên Colab
*Bấm nút **Choose Files** để tải ảnh toàn thân của bạn lên (ví dụ `quan1.jpg`). Nếu không có ảnh, hệ thống sẽ tự dùng ảnh mẫu `demo_sample.jpg` có sẵn trong dự án.*

In [ ]:
import os
from google.colab import files

input_image_path = "/content/C1SE.12-FitVision3D/demo_sample.jpg"

print("Tải ảnh của bạn lên (nhấn Hủy/Cancel nếu muốn dùng ảnh mẫu demo_sample.jpg):")
uploaded = files.upload()
for fn in uploaded.keys():
    input_image_path = os.path.abspath(fn)
    print(f"[✓] Đã nhận ảnh người dùng: {input_image_path}")
    break

print(f"Ảnh được chọn để chạy: {input_image_path}")

### 🟢 BƯỚC 5: Chạy trích xuất số đo may mặc & Tối ưu hóa hình thể 3D
Tùy chỉnh các thông số:
- `--gender`: `male` (nam) hoặc `female` (nữ)
- `--height_cm`: Chiều cao thực tế (cm)
- `--weight_kg`: Cân nặng thực tế (kg)
- `--save_mesh`: Đường dẫn xuất file mô hình 3D (.obj)

In [ ]:
%cd /content/C1SE.12-FitVision3D/ai-service

!/opt/py310/bin/python run_measurement.py \
    --image "$input_image_path" \
    --gender male \
    --height_cm 170.0 \
    --weight_kg 65.0 \
    --save_mesh output_avatar.obj \
    --output_json output_measurements.json

### 🟢 BƯỚC 6: Xem kết quả và tải file Avatar 3D (.obj) + JSON về máy

In [ ]:
import json
from google.colab import files

json_file = '/content/C1SE.12-FitVision3D/ai-service/output_measurements.json'
with open(json_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

print("=" * 60)
print("           KẾT QUẢ TRÍCH XUẤT SỐ ĐO MAY MẶC (ISO 8559)")
print("=" * 60)
info = data.get('thong_tin_nhap', {})
print(f"Giới tính             : {info.get('gioi_tinh')}")
print(f"Chiều cao nhập vào    : {info.get('chieu_cao_nhap_cm')} cm")
print(f"Cân nặng nhập vào     : {info.get('can_nang_nhap_kg')} kg")
print(f"Mật độ mô sinh học    : {info.get('mat_do_co_the_kg_m3')} kg/m³ (Siri-Deurenberg)")
print(f"Cân nặng lưới 3D thật : {info.get('can_nang_luoi_3d_kg')} kg")
print(f"Tỉ lệ scale chuẩn hóa : {info.get('scale_factor_hieu_chinh')}")
print("-" * 60)
for k, v in data.get('so_do_may_mac_cm', {}).items():
    print(f"- {k:22}: {v} cm")
print("=" * 60)

# Tải file JSON và file mô hình 3D (.obj) về máy tính
files.download(json_file)
files.download('/content/C1SE.12-FitVision3D/ai-service/output_avatar.obj')